In [ ]:
from datasets import Dataset
from src.retrival_methods import retrival_pipeline 
from dotenv import load_dotenv
load_dotenv()
import os
from langchain_huggingface import HuggingFaceEmbeddings
from ragas.embeddings import HuggingFaceEmbeddings as RagasHuggingFaceEmbeddings
from langchain.chat_models import init_chat_model
from src.models import build_llms
from ragas.run_config import RunConfig
from ragas.llms import llm_factory
from src.models import get_embedding_model
from openai import OpenAI
import re

import sys
import types

vertex_module = types.ModuleType(
    "langchain_community.chat_models.vertexai"
)

class ChatVertexAI:
    pass

vertex_module.ChatVertexAI = ChatVertexAI

sys.modules[
    "langchain_community.chat_models.vertexai"
] = vertex_module
#making the llm_with_fallback
llm_grok, llm_openrouter, lazy_llm=build_llms()

client = OpenAI(api_key="pranshu123", base_url="http://localhost:8005/v1")
evaluator_llm = llm_factory("Qwen/Qwen3-14B-AWQ", client=client, adapter="litellm")
run_config = RunConfig(timeout=180, max_retries=5,max_wait=60,max_workers=10)



def main_retrival_pipeline(query: str,collection="DemoRAG"):
    
    
    pipeline = retrival_pipeline(collection=collection)
    all_retrieval_results = pipeline.multiquery_RRM(query)

    fused_results = pipeline.reciprocal_rank_fusion(
        all_retrieval_results, k=60, verbose=False
    )
    
    reranked_docs_c = pipeline.reranker_chunks()
    
    response = pipeline.generate_final_answer(chunks=reranked_docs_c, query=query)
    
    return response,reranked_docs_c
questions = [
    "What is Easy Build?",
]

ground_truths = [
    "A B2B2C online channel platform.",
]
ragas_rows = []

for question, ground_truth in zip(questions, ground_truths):
    answer,contexts= main_retrival_pipeline(query=question)
    answer=re.sub(r"<think>.*?</think>","",answer,flags=re.DOTALL,).strip()
    ragas_rows.append({
        "user_input":question,
        "retrieved_contexts":[docs.page_content for docs in contexts],
        "response": answer,
        "reference":ground_truth,
    }
    )


from ragas import EvaluationDataset

evaluation_dataset = EvaluationDataset.from_list(ragas_rows)

ModuleNotFoundError: No module named 'langchain_community.chat_models.vertexai'

In [50]:
evaluator_embeddings = RagasHuggingFaceEmbeddings(model="BAAI/bge-small-en-v1.5",device="cpu")


from ragas import evaluate
from ragas.metrics.collections import (
    AnswerCorrectness,
    Faithfulness,
    ContextPrecision,
    ContextRecall,
    AnswerRelevancy
)

import json

answer_correctness = AnswerCorrectness(
    llm=evaluator_llm,
    embeddings=evaluator_embeddings,
)

answer_relevancy = AnswerRelevancy(
    llm=evaluator_llm,
    embeddings=evaluator_embeddings,
    strictness=1,
)

faithfulness = Faithfulness(
    llm=evaluator_llm,
)

context_precision = ContextPrecision(
    llm=evaluator_llm,
)

context_recall = ContextRecall(
    llm=evaluator_llm,
)


import traceback

try:
    scores = evaluate(
        evaluation_dataset,
        metrics=[
            answer_correctness,
            answer_relevancy,
            faithfulness,
            context_precision,
            context_recall,
        ],
        llm=evaluator_llm,
        embeddings=evaluator_embeddings,
        run_config=run_config,
    )
except Exception:
    traceback.print_exc()

aggregate_scores = dict(scores)

scores_df = scores.to_pandas()
print(scores_df)
per_row_scores = scores_df.to_dict(orient="records")

merged =[]

for row,score_row  in zip(ragas_rows,per_row_scores):
    merged.append({**row,**score_row})

output = {
    "aggregate_scores":aggregate_scores,
    "results":merged,
}

with open("eval_results.json",'w') as f :
    json.dump(output,f,indent=2,ensure_ascii=False )
print(f"Saved {len(merged)} results to eval_resuls.json")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Traceback (most recent call last):
  File "/tmp/ipykernel_2248989/1025193703.py", line 42, in <module>
    scores = evaluate(
             ^^^^^^^^^
  File "/mnt/hdd/Development Pratice/RAG Industry Level/.venv/lib/python3.12/site-packages/ragas/_analytics.py", line 278, in wrapper
    track(IsCompleteEvent(event_type=func.__name__, is_completed=True))
             ^^^^^^^^^^^^^^^^^^^^^
  File "/mnt/hdd/Development Pratice/RAG Industry Level/.venv/lib/python3.12/site-packages/ragas/evaluation.py", line 484, in evaluate
  File "/mnt/hdd/Development Pratice/RAG Industry Level/.venv/lib/python3.12/site-packages/ragas/async_utils.py", line 156, in run
    return asyncio.run(coro)
           ^^^^^^^^^^^^^^^^^
  File "/mnt/hdd/Development Pratice/RAG Industry Level/.venv/lib/python3.12/site-packages/nest_asyncio.py", line 30, in run
    return loop.run_until_complete(task)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/mnt/hdd/Development Pratice/RAG Industry Level/.venv/lib/python3.12/si

NameError: name 'scores' is not defined

In [ ]:
from ragas.metrics.base import Metric

metrics = [
    answer_correctness,
    answer_relevancy,
    faithfulness,
    context_precision,
    context_recall,
]

for i, m in enumerate(metrics):
    print(f"{i}:")
    print("type:", type(m))
    print("is Metric:", isinstance(m, Metric))
    print()

0:
type: <class 'ragas.metrics.collections.answer_correctness.metric.AnswerCorrectness'>
is Metric: False

1:
type: <class 'ragas.metrics.collections.answer_relevancy.metric.AnswerRelevancy'>
is Metric: False

2:
type: <class 'ragas.metrics.collections.faithfulness.metric.Faithfulness'>
is Metric: False

3:
type: <class 'ragas.metrics.collections.context_precision.metric.ContextPrecision'>
is Metric: False

4:
type: <class 'ragas.metrics.collections.context_recall.metric.ContextRecall'>
is Metric: False



In [ ]:
import ragas
import ragas.metrics.base as base
import inspect

print("ragas:", ragas.__version__)
print("Metric class:", base.Metric)
print("Metric file:", inspect.getfile(base.Metric))

ModuleNotFoundError: No module named 'langchain_community.chat_models.vertexai'

In [ ]:
from ragas.metrics.collections import Faithfulness
from ragas.metrics.base import Metric

print(issubclass(Faithfulness, Metric))
print(Faithfulness.__mro__)

False
(<class 'ragas.metrics.collections.faithfulness.metric.Faithfulness'>, <class 'ragas.metrics.collections.base.BaseMetric'>, <class 'ragas.metrics.base.SimpleBaseMetric'>, <class 'ragas.metrics.validators.NumericValidator'>, <class 'ragas.metrics.validators.BaseValidator'>, <class 'abc.ABC'>, <class 'object'>)


In [ ]:
import inspect
from ragas.metrics.collections import Faithfulness

print(inspect.getfile(Faithfulness))

/mnt/hdd/Development Pratice/RAG Industry Level/.venv/lib/python3.12/site-packages/ragas/metrics/collections/faithfulness/metric.py


In [ ]:
print(rows)

[{'question': 'What is Easy Build?', 'contexts': ['Table of Contents\n\n1. Founding Story & Vision 2. B2B2C Business Model & Platform Expansion 3. Department Breakdown & Governance 4. Digital Platform Roadmap & Future Growth\n\n1. Founding Story & Vision\n\nEASY BUILD was founded in 2021 with the vision of organizing the highly fragmented Indian construction and building-materials supply chain. Historically, homeowners and small contractors struggled with fluctuating prices, inconsistent material quality, and unreliable delivery schedules. Our platform integrates physical experience centers with digital commerce, creating transparency and trust.', '4. Digital Platform Roadmap & Future Growth\n\nOver the next five years, EASY BUILD will enhance its AI integrations for price forecasting, automated route optimization, and multimodal customer support chatbots. These systems run on PostgreSQL and local vector databases to drive localized supply-chain logistics.', 'EASY BUILD: CORPORATE HIST